In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
model_dir = "../artifacts/models"
result_dir = "../artifacts/model_results"

os.makedirs(model_dir,exist_ok=True)

os.makedirs(result_dir,exist_ok=True)

In [3]:
file_path = "../data/processed/cleaned_customer_churn.csv"
df = pd.read_csv(file_path)

print("Dataset Shape:")
df.shape

Dataset Shape:


(7043, 21)

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")

df["TotalCharges"] = (df["TotalCharges"].fillna(0))

In [5]:
if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({"No": 0,"Yes": 1})


print("Churn Distribution:")
print(df["Churn"].value_counts())

print("Churn Percentage:")
print(df["Churn"].value_counts(normalize=True).mul(100))

Churn Distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64
Churn Percentage:
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [6]:
customer_ids = df["customerID"].copy()

In [10]:
X = df.drop(columns=["customerID","Churn"])
y = df["Churn"]

print("Features Shape:")
print(X.shape)

print("Target Shape:")
print(y.shape)

Features Shape:
(7043, 19)
Target Shape:
(7043,)


In [11]:
numerical_features = (X.select_dtypes(include=["int64", "float64"]).columns.tolist())

categorical_features = (X.select_dtypes(include=["object"]).columns.tolist())

print("Numerical Features:")
print(numerical_features)

print("Categorical Features:")
print(categorical_features)

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test,id_train,id_test= train_test_split(X,y,customer_ids,test_size=0.20,random_state=42,stratify=y)


print("Training Shape:")
print(X_train.shape)
print()
print("Testing Shape:")
print(X_test.shape)

Training Shape:
(5634, 19)

Testing Shape:
(1409, 19)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
def create_preprocessor():
    numerical_pipeline = Pipeline(steps=[("imputer",SimpleImputer(strategy="median"))])

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot",OneHotEncoder(handle_unknown="ignore",drop="first"))
        ])

    preprocessor = ColumnTransformer(transformers=[
            ("numerical",numerical_pipeline,numerical_features),
            ("categorical",categorical_pipeline,categorical_features)
        ])

    return preprocessor

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
models = {
    "Logistic Regression": Pipeline(steps=[
            ("preprocessor",create_preprocessor()),
            ("scaler",StandardScaler(with_mean=False)),
            ("classifier",LogisticRegression(max_iter=1000,random_state=42))
        ]),
    "Decision Tree": Pipeline(steps=[
            ("preprocessor",create_preprocessor()),
            ("classifier",DecisionTreeClassifier(random_state=42))
        ]),
    "Random Forest": Pipeline(steps=[
            ("preprocessor",create_preprocessor()),
            ("classifier",RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1))
        ])
}


In [21]:
from sklearn.metrics import f1_score,recall_score,precision_score,accuracy_score,roc_auc_score,classification_report
def evaluate_model(model_name,model,X_train,X_test,y_train,y_test):

    print(f"Training: {model_name}")

    # Train model
    model.fit(X_train,y_train)

    # Class prediction
    y_pred = model.predict(X_test)

    # Probability of churn
    y_probability = (model.predict_proba(X_test)[:, 1])

    # Metrics
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)
    roc_auc = roc_auc_score(y_test,y_probability)

    print(f"Accuracy  : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall : {recall:.4f}")

    print(f"F1 Score  : {f1:.4f}")

    print(f"ROC-AUC   : {roc_auc:.4f}")

    print("Classification Report:")

    print(classification_report(y_test,y_pred))

    result = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc
    }

    return result

In [22]:
results = []

for model_name, model in models.items():
    result = evaluate_model(model_name,model,X_train,X_test,y_train,y_test)

    results.append(result)


Training: Logistic Regression
Accuracy  : 0.8070
Precision : 0.6594
Recall : 0.5642
F1 Score  : 0.6081
ROC-AUC   : 0.8419
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Training: Decision Tree
Accuracy  : 0.7260
Precision : 0.4838
Recall : 0.4786
F1 Score  : 0.4812
ROC-AUC   : 0.6465
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.82      0.81      1035
           1       0.48      0.48      0.48       374

    accuracy                           0.73      1409
   macro avg       0.65      0.65      0.65      1409
weighted avg       0.73      0.73      0.73      1409

Training: Random Forest
Accuracy  : 0.7942
Precision : 0.6382


In [20]:
results_df = pd.DataFrame(results)
results_df = (results_df.sort_values("F1_Score",ascending=False).reset_index(drop=True))

print("BASELINE MODEL COMPARISON")
print()
print(results_df.round(4))


BASELINE MODEL COMPARISON

                 Model  Accuracy  Precision  Recall  F1_Score  ROC_AUC
0  Logistic Regression    0.8070     0.6594  0.5642    0.6081   0.8419
1        Random Forest    0.7942     0.6382  0.5187    0.5723   0.8272
2        Decision Tree    0.7260     0.4838  0.4786    0.4812   0.6465


In [23]:
results_df.to_csv(os.path.join(result_dir,"baseline_model_comparison.csv"),index=False)

In [24]:
feature_preprocessor = (create_preprocessor())

X_train_processed = (feature_preprocessor.fit_transform(X_train))



In [25]:
feature_names = (feature_preprocessor.get_feature_names_out())

print("Number of Features After Encoding:")
print(len(feature_names))

Number of Features After Encoding:
30


In [26]:
from sklearn.feature_selection import SelectKBest,f_classif
feature_selector = SelectKBest(score_func=f_classif,k="all")

feature_selector.fit(X_train_processed,y_train)

SelectKBest(k='all')

In [28]:
feature_scores = pd.DataFrame({
        "Feature": feature_names,
        "Score": feature_selector.scores_,
        "P_Value": feature_selector.pvalues_
    })

feature_scores = (feature_scores.sort_values("Score",ascending=False).reset_index(drop=True))


print("Top Important Features According to ANOVA F-Test:")
print(feature_scores.head(20))

Top Important Features According to ANOVA F-Test:
                                              Feature       Score  \
0                                   numerical__tenure  763.890183   
1            categorical__InternetService_Fiber optic  610.199922   
2         categorical__PaymentMethod_Electronic check  595.424491   
3                      categorical__Contract_Two year  566.070664   
4    categorical__StreamingMovies_No internet service  311.490238   
5        categorical__TechSupport_No internet service  311.490238   
6   categorical__DeviceProtection_No internet service  311.490238   
7       categorical__OnlineBackup_No internet service  311.490238   
8     categorical__OnlineSecurity_No internet service  311.490238   
9                     categorical__InternetService_No  311.490238   
10       categorical__StreamingTV_No internet service  311.490238   
11                          numerical__MonthlyCharges  229.903823   
12                  categorical__PaperlessBilling_Yes

In [29]:
feature_scores.to_csv(os.path.join(result_dir,"feature_selection_scores.csv"),index=False)

In [30]:
tuning_pipeline = Pipeline(steps=[
        ("preprocessor",create_preprocessor()),
        ("selector",SelectKBest(score_func=f_classif,k="all")),
        ("scaler",StandardScaler(with_mean=False)),
        ("classifier",LogisticRegression(max_iter=2000,random_state=42))
    ])

In [31]:
parameter_grid = {
    "selector__k": [15,20,25,"all"],
    "classifier__C": [0.01,0.1,1,10],
    "classifier__class_weight": [None,"balanced"]
}

In [32]:
from sklearn.model_selection import StratifiedKFold
cross_validation = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [33]:
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=parameter_grid,
    scoring="f1",
    cv=cross_validation,
    n_jobs=-1,
    verbose=1,
    refit=True
)

In [35]:
print("Starting Hyperparameter Tuning :")

grid_search.fit(X_train,y_train)
print("Hyperparameter Tuning Completed.")


Starting Hyperparameter Tuning :
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Hyperparameter Tuning Completed.


In [36]:
print("Best Parameters:")
print(grid_search.best_params_)

print("Best Cross-Validation F1 Score:")
print(round(grid_search.best_score_,4))


Best Parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'selector__k': 'all'}
Best Cross-Validation F1 Score:
0.6293


In [37]:
best_model = (grid_search.best_estimator_)

In [38]:
y_pred_best = (best_model.predict(X_test))
y_probability_best = (best_model.predict_proba(X_test)[:, 1])

In [39]:
final_accuracy = accuracy_score(y_test,y_pred_best)
final_precision = precision_score(y_test,y_pred_best)
final_recall = recall_score(y_test,y_pred_best)

final_f1 = f1_score(y_test,y_pred_best)
final_roc_auc = roc_auc_score(y_test,y_probability_best)


In [40]:
print("TUNED LOGISTIC REGRESSION RESULTS :")
print(f"Accuracy  : {final_accuracy:.4f}")
print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1 Score  : {final_f1:.4f}")
print(f"ROC-AUC   : {final_roc_auc:.4f}")
print("Classification Report:")
print(classification_report(y_test,y_pred_best))


TUNED LOGISTIC REGRESSION RESULTS :
Accuracy  : 0.7395
Precision : 0.5060
Recall    : 0.7834
F1 Score  : 0.6149
ROC-AUC   : 0.8415
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [41]:
final_result = pd.DataFrame([
        {"Model":"Tuned Logistic Regression",
        "Accuracy":final_accuracy,
        "Precision":final_precision,
        "Recall":final_recall,
        "F1_Score":final_f1,
        "ROC_AUC":final_roc_auc
        }
    ])

print("Final Tuned Model Result:")
print(final_result.round(4))


Final Tuned Model Result:
                       Model  Accuracy  Precision  Recall  F1_Score  ROC_AUC
0  Tuned Logistic Regression    0.7395      0.506  0.7834    0.6149   0.8415


In [42]:
final_result.to_csv(os.path.join(result_dir,"tuned_model_result.csv"),index=False)

In [43]:
prediction_results = pd.DataFrame(
    {
        "customerID":id_test.values,
        "Actual_Churn":y_test.values,
        "Predicted_Churn":y_pred_best,
        "Churn_Probability":y_probability_best
    }
)

prediction_results = (prediction_results.sort_values("Churn_Probability",ascending=False))
prediction_results.to_csv(os.path.join(result_dir,"test_customer_predictions.csv"),index=False)

In [44]:
import joblib 
model_path = os.path.join(model_dir,"best_churn_model.pkl")

joblib.dump(best_model,model_path)

print("Best model saved at:")
print(model_path)

Best model saved at:
../artifacts/models\best_churn_model.pkl
